# 7장. 트랜스포머 (Transformer)
**파이토치 트랜스포머를 활용한 자연어 처리와 컴퓨터비전 심층학습**

---
이 노트북은 7장의 예제 코드를 순서대로 정리한 것입니다.

| 예제 | 내용 |
|------|------|
| 예제 7.1 | 위치 인코딩 |
| 예제 7.2 | 데이터셋 다운로드 및 전처리 |
| 예제 7.3 | 트랜스포머 모델 구성 |
| 예제 7.4 | 트랜스포머 모델 구조 |
| 예제 7.5 | 배치 데이터 생성 |
| 예제 7.6 | 어텐션 마스크 생성 |
| 예제 7.7 | 모델 학습 및 평가 |
| 예제 7.8 | 트랜스포머 모델 번역 결과 |

## 사전 준비: 라이브러리 설치

아래 셀을 실행해 필요한 패키지를 설치합니다.

In [ ]:
# 필요한 패키지 설치
!pip install torchdata torchtext portalocker spacy
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

---
## 예제 7.1 — 위치 인코딩 (Positional Encoding)

트랜스포머는 입력 시퀀스를 병렬로 처리하기 때문에 단어의 순서 정보를 별도로 추가해야 합니다.  
`sin` / `cos` 함수를 이용한 위치 인코딩을 구현하고 시각화합니다.

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
import math
import torch
from torch import nn
from matplotlib import pyplot as plt


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)  # 짝수 차원: sin
        pe[:, 0, 1::2] = torch.cos(position * div_term)  # 홀수 차원: cos
        self.register_buffer("pe", pe)  # 학습 파라미터로 등록하지 않음

    def forward(self, x):
        x = x + self.pe[: x.size(0)]
        return self.dropout(x)


# 위치 인코딩 시각화 (d_model=128, max_len=50)
encoding = PositionalEncoding(d_model=128, max_len=50)

plt.pcolormesh(encoding.pe.numpy().squeeze(), cmap="RdBu")
plt.xlabel("Embedding Dimension")
plt.xlim((0, 128))
plt.ylabel("Position")
plt.colorbar()
plt.title("Positional Encoding")
plt.show()

---
## 예제 7.2 — 데이터셋 다운로드 및 전처리

**Multi30k** 데이터셋(영어–독일어 병렬 말뭉치, 약 30,000개)을 불러오고  
spaCy 토크나이저와 어휘 사전을 생성합니다.

In [ ]:
from torchtext.datasets import Multi30k
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

# 소스(독일어) / 타깃(영어) 설정
SRC_LANGUAGE = "de"
TGT_LANGUAGE = "en"

# 특수 토큰 인덱스
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ["<unk>", "<pad>", "<bos>", "<eos>"]

# spaCy 토크나이저 로드
token_transform = {
    SRC_LANGUAGE: get_tokenizer("spacy", language="de_core_news_sm"),
    TGT_LANGUAGE: get_tokenizer("spacy", language="en_core_web_sm"),
}
print("Token Transform:")
print(token_transform)


def generate_tokens(text_iter, language):
    """데이터 이터레이터에서 토큰을 생성하는 제너레이터"""
    language_index = {SRC_LANGUAGE: 0, TGT_LANGUAGE: 1}
    for text in text_iter:
        yield token_transform[language](text[language_index[language]])


# 언어별 어휘 사전 생성
vocab_transform = {}
for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    train_iter = Multi30k(split="train", language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
    vocab_transform[language] = build_vocab_from_iterator(
        generate_tokens(train_iter, language),
        min_freq=1,
        specials=special_symbols,
        special_first=True,
    )

for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    vocab_transform[language].set_default_index(UNK_IDX)

print("\nVocab Transform:")
print(vocab_transform)

---
## 예제 7.3 — 트랜스포머 모델 구성

토큰 임베딩(`TokenEmbedding`)과 Seq2Seq 트랜스포머(`Seq2SeqTransformer`) 클래스를 정의합니다.

In [ ]:
import math
import torch
from torch import nn


# ── 위치 인코딩 (예제 7.1과 동일) ──────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[: x.size(0)]
        return self.dropout(x)


# ── 토큰 임베딩 ────────────────────────────────────────────────
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.emb_size = emb_size

    def forward(self, tokens):
        # 임베딩 값을 sqrt(d_model)로 스케일링
        return self.embedding(tokens.long()) * math.sqrt(self.emb_size)


# ── Seq2Seq 트랜스포머 ─────────────────────────────────────────
class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        num_encoder_layers,
        num_decoder_layers,
        emb_size,
        max_len,
        nhead,
        src_vocab_size,
        tgt_vocab_size,
        dim_feedforward,
        dropout=0.1,
    ):
        super().__init__()
        self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(
            d_model=emb_size, max_len=max_len, dropout=dropout
        )
        self.transformer = nn.Transformer(
            d_model=emb_size,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )
        # 어휘 사전 크기의 로짓(logit) 생성
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(
        self,
        src,
        trg,
        src_mask,
        tgt_mask,
        src_padding_mask,
        tgt_padding_mask,
        memory_key_padding_mask,
    ):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(trg))
        outs = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            memory_mask=None,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )
        return self.generator(outs)

    def encode(self, src, src_mask):
        return self.transformer.encoder(
            self.positional_encoding(self.src_tok_emb(src)), src_mask
        )

    def decode(self, tgt, memory, tgt_mask):
        return self.transformer.decoder(
            self.positional_encoding(self.tgt_tok_emb(tgt)), memory, tgt_mask
        )

---
## 예제 7.4 — 트랜스포머 모델 선언 및 구조 확인

`Seq2SeqTransformer`를 인스턴스화하고 손실 함수, 옵티마이저를 설정합니다.

In [ ]:
from torch import optim

BATCH_SIZE = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {DEVICE}")

model = Seq2SeqTransformer(
    num_encoder_layers=3,
    num_decoder_layers=3,
    emb_size=512,
    max_len=512,
    nhead=8,
    src_vocab_size=len(vocab_transform[SRC_LANGUAGE]),
    tgt_vocab_size=len(vocab_transform[TGT_LANGUAGE]),
    dim_feedforward=512,
).to(DEVICE)

# PAD 토큰은 손실 계산에서 무시
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX).to(DEVICE)
optimizer = optim.Adam(model.parameters())

# 모델 구조 출력
for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("  └", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("  |  └", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("  |  |  └", sssub_name)

---
## 예제 7.5 — 배치 데이터 생성

텍스트 전처리 파이프라인(토크나이저 → 어휘 인덱싱 → BOS/EOS 추가)과  
`DataLoader`를 구성합니다.

In [ ]:
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence


def sequential_transforms(*transforms):
    """여러 전처리 함수를 순서대로 적용하는 함수를 반환"""
    def func(txt_input):
        for transform in transforms:
            txt_input = transform(txt_input)
        return txt_input
    return func


def input_transform(token_ids):
    """BOS / EOS 특수 토큰 추가"""
    return torch.cat(
        (torch.tensor([BOS_IDX]), torch.tensor(token_ids), torch.tensor([EOS_IDX]))
    )


def collator(batch):
    """배치 내 시퀀스를 PAD로 맞춰 텐서로 변환"""
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        src_batch.append(text_transform[SRC_LANGUAGE](src_sample.rstrip("\n")))
        tgt_batch.append(text_transform[TGT_LANGUAGE](tgt_sample.rstrip("\n")))
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX)
    return src_batch, tgt_batch


# 텍스트 전처리 파이프라인: 토크나이저 → 어휘 인덱싱 → BOS/EOS 추가
text_transform = {}
for language in [SRC_LANGUAGE, TGT_LANGUAGE]:
    text_transform[language] = sequential_transforms(
        token_transform[language],
        vocab_transform[language],
        input_transform,
    )

# 검증 데이터로 배치 확인
data_iter = Multi30k(split="valid", language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
dataloader = DataLoader(data_iter, batch_size=BATCH_SIZE, collate_fn=collator)
source_tensor, target_tensor = next(iter(dataloader))

print("(source, target):")
print(next(iter(data_iter)))
print("\nsource_batch:", source_tensor.shape)
print(source_tensor)
print("\ntarget_batch:", target_tensor.shape)
print(target_tensor)

---
## 예제 7.6 — 어텐션 마스크 생성

- `generate_square_subsequent_mask`: 디코더 인과성 마스크 (상삼각행렬, 미래 토큰 차단)
- `create_mask`: 소스/타깃 마스크 및 패딩 마스크를 한 번에 생성

In [ ]:
def generate_square_subsequent_mask(s):
    """s×s 크기의 인과성 마스크 생성 (상삼각행렬, 0=-inf / 1=0.0)"""
    mask = (torch.triu(torch.ones((s, s), device=DEVICE)) == 1).transpose(0, 1)
    mask = (
        mask.float()
        .masked_fill(mask == 0, float("-inf"))  # 미래 위치 → -inf
        .masked_fill(mask == 1, float(0.0))     # 현재/과거 위치 → 0
    )
    return mask


def create_mask(src, tgt):
    """소스/타깃 마스크 및 패딩 마스크를 생성"""
    src_seq_len = src.shape[0]
    tgt_seq_len = tgt.shape[0]

    # 타깃 시퀀스의 인과성 마스크
    tgt_mask = generate_square_subsequent_mask(tgt_seq_len)
    # 소스 마스크는 모두 False (모든 소스 토큰 참조 허용)
    src_mask = torch.zeros((src_seq_len, src_seq_len), device=DEVICE).type(torch.bool)

    # 패딩 토큰 위치를 True로 표시
    src_padding_mask = (src == PAD_IDX).transpose(0, 1)
    tgt_padding_mask = (tgt == PAD_IDX).transpose(0, 1)
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask


# 마스크 생성 확인
target_input  = target_tensor[:-1, :]  # 마지막 토큰 제외 (디코더 입력)
target_out    = target_tensor[1:,  :]  # 첫 번째 토큰 제외 (예측 대상)

source_mask, target_mask, source_padding_mask, target_padding_mask = create_mask(
    source_tensor, target_input
)

print("source_mask:", source_mask.shape)
print(source_mask)
print("\ntarget_mask:", target_mask.shape)
print(target_mask)
print("\nsource_padding_mask:", source_padding_mask.shape)
print(source_padding_mask)
print("\ntarget_padding_mask:", target_padding_mask.shape)
print(target_padding_mask)

---
## 예제 7.7 — 모델 학습 및 평가

학습/검증 루프를 구현합니다. 5 에폭 동안 학습하며 Train loss와 Val loss를 출력합니다.

In [ ]:
def run(model, optimizer, criterion, split):
    """학습(split='train') 또는 평가(split='valid') 수행 후 평균 손실 반환"""
    model.train() if split == "train" else model.eval()
    data_iter = Multi30k(split=split, language_pair=(SRC_LANGUAGE, TGT_LANGUAGE))
    dataloader = DataLoader(data_iter, batch_size=BATCH_SIZE, collate_fn=collator)

    losses = 0
    for source_batch, target_batch in dataloader:
        source_batch = source_batch.to(DEVICE)
        target_batch = target_batch.to(DEVICE)

        # 디코더 입력/출력 분리 (teacher forcing)
        target_input  = target_batch[:-1, :]
        target_output = target_batch[1:,  :]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(
            source_batch, target_input
        )

        logits = model(
            src=source_batch,
            trg=target_input,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            src_padding_mask=src_padding_mask,
            tgt_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask,
        )

        optimizer.zero_grad()
        loss = criterion(
            logits.reshape(-1, logits.shape[-1]),
            target_output.reshape(-1),
        )
        if split == "train":
            loss.backward()
            optimizer.step()
        losses += loss.item()

    return losses / len(list(dataloader))


# 5 에폭 학습
for epoch in range(5):
    train_loss = run(model, optimizer, criterion, "train")
    val_loss   = run(model, optimizer, criterion, "valid")
    print(f"Epoch: {epoch+1}, Train loss: {train_loss:.3f}, Val loss: {val_loss:.3f}")

---
## 예제 7.8 — 트랜스포머 모델 번역 결과

**그리디 디코딩(Greedy Decoding)** 방식으로 독일어 문장을 영어로 번역합니다.  
현재 시점에서 가장 확률이 높은 단어를 선택해 순차적으로 생성합니다.

In [ ]:
def greedy_decode(model, source_tensor, source_mask, max_len, start_symbol):
    """그리디 디코딩으로 번역 토큰 시퀀스 생성"""
    source_tensor = source_tensor.to(DEVICE)
    source_mask   = source_mask.to(DEVICE)

    # 소스 인코딩
    memory = model.encode(source_tensor, source_mask)
    # 디코더 시작 토큰 (BOS)
    ys = torch.ones(1, 1).fill_(start_symbol).type(torch.long).to(DEVICE)

    for i in range(max_len - 1):
        memory      = memory.to(DEVICE)
        target_mask = generate_square_subsequent_mask(ys.size(0))
        target_mask = target_mask.type(torch.bool).to(DEVICE)

        out  = model.decode(ys, memory, target_mask)
        out  = out.transpose(0, 1)
        prob = model.generator(out[:, -1])

        # 가장 확률 높은 토큰 선택
        _, next_word = torch.max(prob, dim=1)
        next_word    = next_word.item()

        ys = torch.cat(
            [ys, torch.ones(1, 1).type_as(source_tensor.data).fill_(next_word)], dim=0
        )
        if next_word == EOS_IDX:  # EOS 토큰 예측 시 종료
            break

    return ys


def translate(model, source_sentence):
    """독일어 문장을 영어로 번역"""
    model.eval()
    source_tensor = text_transform[SRC_LANGUAGE](source_sentence).view(-1, 1)
    num_tokens    = source_tensor.shape[0]
    src_mask      = (torch.zeros(num_tokens, num_tokens)).type(torch.bool)

    tgt_tokens = greedy_decode(
        model,
        source_tensor,
        src_mask,
        max_len=num_tokens + 5,
        start_symbol=BOS_IDX,
    ).flatten()

    output = vocab_transform[TGT_LANGUAGE].lookup_tokens(
        list(tgt_tokens.cpu().numpy())
    )[1:-1]  # BOS, EOS 제거
    return " ".join(output)


# ── 번역 테스트 ─────────────────────────────────────────────────
# OOV 단어 포함 (Iglu: 어휘 사전에 없는 단어)
output_oov = translate(model, "Eine Gruppe von Menschen steht vor einem Iglu .")
# 정상 단어 (Gebäude: 건물)
output     = translate(model, "Eine Gruppe von Menschen steht vor einem Gebäude .")

print("[OOV 포함] ", output_oov)
print("[정상 입력] ", output)

---
## 참고: 트랜스포머 클래스 파라미터 요약

| 파라미터 | 설명 |
|----------|------|
| `d_model` | 모델의 입·출력 임베딩 차원 크기 |
| `nhead` | 멀티 헤드 어텐션의 헤드 개수 |
| `num_encoder_layers` | 인코더 계층 수 |
| `num_decoder_layers` | 디코더 계층 수 |
| `dim_feedforward` | 순방향 신경망 은닉층 크기 |
| `dropout` | 드롭아웃 비율 |
| `activation` | 순방향 신경망 활성화 함수 |
| `layer_norm_eps` | 레이어 정규화 입실론 값 |